In [57]:
# import parquet
import pandas as pd

In [58]:
# read parquet file
jaccard = pd.read_parquet('../../Data/jaccard_column_pairs.parquet')
lazo = pd.read_parquet('../../Data/lazo_candidates.parquet')
embedding = pd.read_parquet('../../Data/joinability_pairs_embedding_column_level.parquet')

embedding_dataset = pd.read_parquet('../../Data/joinability_pairs_embedding.parquet')

In [44]:
jaccard.head()

,dataset_id_1,column_name_1,dataset_id_2,column_name_2,score,method
0,3rfa-3xsf,longitude,b4mf-rg6h,longitude,1.0,jaccard
1,bmxf-3rd4,nta,ykx2-pdw8,nta,1.0,jaccard
2,cfzn-4iza,bin,xck4-5xd5,bin,1.0,jaccard
3,4te8-9n3p,the_geom,mrjc-v9pm,the_geom,1.0,jaccard
4,irhv-jqz7,bin,p5md-weyf,bin,1.0,jaccard


In [45]:
jaccard.describe()

,score
count,6.180128e+06
mean,5.398650e-01
std,2.735258e-01
min,3.000000e-01
25%,3.333000e-01
50%,3.333000e-01
75%,7.500000e-01
max,1.000000e+00


In [59]:
lazo.describe()

,score
count,430144.000000
mean,0.754841
std,0.216358
min,0.300000
25%,0.578900
50%,0.777800
75%,1.000000
max,1.000000


In [47]:
embedding.describe()

,score
count,675202.000000
mean,0.939259
std,0.054664
min,0.850000
25%,0.883100
50%,0.934200
75%,1.000000
max,1.000000


In [62]:
# ── Normalize & Merge ──────────────────────────────────────────────────
# Root cause: Embedding uses original-case column names ('Time', 'Borough')
# while Jaccard and Lazo use lowercase. Fix: lowercase all column names
# before merging, then dedup keeping the highest score per pair.

KEY = ['dataset_id_1', 'column_name_1', 'dataset_id_2', 'column_name_2']

def normalize(df, score_col):
    df = df[KEY + ['score']].copy()
    # lowercase column names
    df['column_name_1'] = df['column_name_1'].str.lower()
    df['column_name_2'] = df['column_name_2'].str.lower()
    # normalize pair order so (A,x,B,y) and (B,y,A,x) become the same key
    swap = (
        (df['dataset_id_1'] > df['dataset_id_2']) |
        ((df['dataset_id_1'] == df['dataset_id_2']) &
         (df['column_name_1'] > df['column_name_2']))
    )
    df.loc[swap, KEY] = df.loc[swap, ['dataset_id_2','column_name_2',
                                        'dataset_id_1','column_name_1']].values
    # dedup: keep max score per pair (lowercasing may create duplicates)
    df = df.groupby(KEY, as_index=False)['score'].max()
    df = df.rename(columns={'score': score_col})
    return df

j = normalize(jaccard,   'jaccard_score')
l = normalize(lazo,      'lazo_score')
e = normalize(embedding, 'embedding_score')

merged = (
    j.merge(l, on=KEY, how='outer')
     .merge(e, on=KEY, how='outer')
)
merged[['jaccard_score','lazo_score','embedding_score']] = (
    merged[['jaccard_score','lazo_score','embedding_score']].fillna(0.0)
)

print(f'Total unique pairs: {len(merged):,}')
print(merged.head(5).to_string())


Total unique pairs: 6,983,164
  dataset_id_1            column_name_1 dataset_id_2            column_name_2  jaccard_score  lazo_score  embedding_score
0    22rr-ujq3  administrative district    2a67-6eaq           admin district            0.0         0.0           0.9174
1    22rr-ujq3  administrative district    2t89-756w  administrative district            0.0         0.0           1.0000
2    22rr-ujq3  administrative district    2vha-97jm                 district            0.0         0.0           0.8816
3    22rr-ujq3  administrative district    32rn-zwi7                 district            0.0         0.0           0.8816
4    22rr-ujq3  administrative district    48wd-x25j  administrative district            0.0         0.0           1.0000


In [82]:
# Vote: 1 if method found this pair (using each method's pre-filter threshold)
merged['j_vote'] = (merged['jaccard_score']   >= 0.5).astype(int)
merged['l_vote'] = (merged['lazo_score']      >= 0.5).astype(int)
merged['e_vote'] = (merged['embedding_score'] > 0).astype(int)
merged['vote_count'] = merged['j_vote'] + merged['l_vote'] + merged['e_vote']

print('Vote distribution:')
print(merged['vote_count'].value_counts().sort_index())
print()
print('Method agreement breakdown:')
for jv, lv, ev in [(1,1,1),(1,1,0),(1,0,1),(0,1,1),(1,0,0),(0,1,0),(0,0,1)]:
    n = ((merged.j_vote==jv)&(merged.l_vote==lv)&(merged.e_vote==ev)).sum()
    label = '+'.join([m for m,v in [('J',jv),('L',lv),('E',ev)] if v])
    print(f'  {label:8s}: {n:,}')


Vote distribution:
vote_count
0    3334631
1    3381007
2     252160
3      15366
Name: count, dtype: int64

Method agreement breakdown:
  J+L+E   : 15,366
  J+L     : 81,827
  J+E     : 167,580
  L+E     : 2,753
  J       : 2,620,658
  L       : 270,855
  E       : 489,494


In [84]:
import numpy as np

SEED = 42
N_HIGH = 50
N_MID_EACH = 50   # 每個 2-method 組合各抽 50 → medium 共 150
N_LOW = 50        # 3 個 1-method 組合均分 → 各約 17
N_NEG = 50

def sample_stratum(mask, n, label):
    pool = merged[mask]
    s = pool.sample(min(n, len(pool)), random_state=SEED)
    print(f'{label:30s} pool={len(pool):>9,}  sampled={len(s)}')
    return s

# ── High: all 3 agree ────────────────────────────────────────────────────
high = sample_stratum(
    (merged.j_vote==1)&(merged.l_vote==1)&(merged.e_vote==1),
    N_HIGH, 'High (J+L+E)')

# ── Medium: exactly 2 agree, 50 each ────────────────────────────────────
mid_jl = sample_stratum(
    (merged.j_vote==1)&(merged.l_vote==1)&(merged.e_vote==0),
    N_MID_EACH, 'Mid J+L')
mid_je = sample_stratum(
    (merged.j_vote==1)&(merged.l_vote==0)&(merged.e_vote==1),
    N_MID_EACH, 'Mid J+E')
mid_le = sample_stratum(
    (merged.j_vote==0)&(merged.l_vote==1)&(merged.e_vote==1),
    N_MID_EACH, 'Mid L+E')
mid = pd.concat([mid_jl, mid_je, mid_le], ignore_index=True)

# ── Low: exactly 1 agrees, ~17 each (3 groups split N_LOW) ──────────────
n_each_low = N_LOW // 3
low_j = sample_stratum(
    (merged.j_vote==1)&(merged.l_vote==0)&(merged.e_vote==0),
    n_each_low, 'Low J only')
low_l = sample_stratum(
    (merged.j_vote==0)&(merged.l_vote==1)&(merged.e_vote==0),
    n_each_low, 'Low L only')
low_e = sample_stratum(
    (merged.j_vote==0)&(merged.l_vote==0)&(merged.e_vote==1),
    N_LOW - 2*n_each_low, 'Low E only')
low = pd.concat([low_j, low_l, low_e], ignore_index=True)

# ── Negative: 0 methods agree, random pairs not in any output ────────────
with open('../../Data/nyc_socrata_datasets.json') as f:
    raw_data = json.load(f)

all_cols = [
    (d['id'], col.lower())
    for d in raw_data
    for col in d.get('columns', [])
]
merged_keys = set(
    zip(merged.dataset_id_1, merged.column_name_1,
        merged.dataset_id_2, merged.column_name_2)
)
rng = np.random.default_rng(SEED)
neg_rows, attempts = [], 0
while len(neg_rows) < N_NEG and attempts < 500_000:
    idx_a, idx_b = rng.choice(len(all_cols), size=2, replace=False)
    ds_a, col_a = all_cols[idx_a]
    ds_b, col_b = all_cols[idx_b]
    if ds_a == ds_b:
        attempts += 1; continue
    if (ds_a, col_a) > (ds_b, col_b):
        ds_a, col_a, ds_b, col_b = ds_b, col_b, ds_a, col_a
    if (ds_a, col_a, ds_b, col_b) in merged_keys:
        attempts += 1; continue
    neg_rows.append({
        'dataset_id_1': ds_a, 'column_name_1': col_a,
        'dataset_id_2': ds_b, 'column_name_2': col_b,
        'jaccard_score': 0.0, 'lazo_score': 0.0, 'embedding_score': 0.0,
        'j_vote': 0, 'l_vote': 0, 'e_vote': 0, 'vote_count': 0,
    })
    attempts += 1
neg = pd.DataFrame(neg_rows)
print(f'Negative (random)              pool=        —  sampled={len(neg)}')

# ── Combine + dedup safety check ─────────────────────────────────────────
sample = pd.concat([high, mid, low, neg], ignore_index=True)
before = len(sample)
sample = sample.drop_duplicates(subset=KEY).reset_index(drop=True)
print(f'\nTotal: {len(sample)} (dropped {before - len(sample)} duplicates)')

sample['confidence'] = sample['vote_count'].map(
    {3: 'high', 2: 'medium', 1: 'low', 0: 'negative'})
print(sample['confidence'].value_counts())


High (J+L+E)                   pool=   15,366  sampled=50
Mid J+L                        pool=   81,827  sampled=50
Mid J+E                        pool=  167,580  sampled=50
Mid L+E                        pool=    2,753  sampled=50
Low J only                     pool=2,620,658  sampled=16
Low L only                     pool=  270,855  sampled=16
Low E only                     pool=  489,494  sampled=18
Negative (random)              pool=        —  sampled=50

Total: 300 (dropped 0 duplicates)
confidence
medium      150
high         50
low          50
negative     50
Name: count, dtype: int64


In [85]:
# Attach up to 5 sample values per column for human annotation

# Build lookup: (dataset_id, col_name_lower) -> [values]
# Use (d.get("sample_rows") or []) to handle datasets where sample_rows is None
sample_val_lookup = {}
for d in raw_data:
    ds_id = d["id"]
    for row in (d.get("sample_rows") or [])[:20]:
        for k, v in row.items():
            key = (ds_id, k.lower())
            if key not in sample_val_lookup:
                sample_val_lookup[key] = set()
            if v is not None and str(v).strip():
                sample_val_lookup[key].add(str(v).strip())

def get_samples(ds_id, col_name, n=5):
    vals = sample_val_lookup.get((ds_id, col_name.lower()), set())
    return ", ".join(sorted(vals)[:n]) if vals else ""

sample["sample_values_1"] = sample.apply(
    lambda r: get_samples(r.dataset_id_1, r.column_name_1), axis=1)
sample["sample_values_2"] = sample.apply(
    lambda r: get_samples(r.dataset_id_2, r.column_name_2), axis=1)

# Final column order
out_cols = [
    "dataset_id_1", "column_name_1", "sample_values_1",
    "dataset_id_2", "column_name_2", "sample_values_2",
    "jaccard_score", "lazo_score", "embedding_score",
    "vote_count", "confidence",
]
ground_truth = sample[out_cols].reset_index(drop=True)
ground_truth.index.name = "pair_id"

out_path = "../../Data/ground_truth_candidates.csv"
ground_truth.to_csv(out_path)
print(f"Saved {len(ground_truth)} pairs -> {out_path}")
print()
print(ground_truth.head(3).to_string())


Saved 300 pairs -> ../../Data/ground_truth_candidates.csv

        dataset_id_1 column_name_1                                    sample_values_1 dataset_id_2 column_name_2                                    sample_values_2  jaccard_score  lazo_score  embedding_score  vote_count confidence
pair_id                                                                                                                                                                                                                               
0          i4gi-tjb9       borough  Bronx, Brooklyn, Manhattan, Queens, Staten Island    kb2e-tjy3       borough  Bronx, Brooklyn, Manhattan, Queens, Staten Island            1.0        1.00           1.0000           3       high
1          ci36-d7ea  survey_pp_se                       0.51, 0.64, 0.65, 0.66, 0.69    kkng-ugna  survey_pp_ct                        0.64, 0.66, 0.78, 0.8, 0.86            1.0        0.76           0.8543           3       high
2          phvi-d

In [86]:
ground_truth['embedding_score']

pair_id
0      1.0000
1      0.8543
2      1.0000
3      1.0000
4      1.0000
        ...  
295    0.0000
296    0.0000
297    0.0000
298    0.0000
299    0.0000
Name: embedding_score, Length: 300, dtype: float64

In [87]:
ground_truth[ground_truth['confidence'] == 'high']

,dataset_id_1,column_name_1,sample_values_1,dataset_id_2,column_name_2,sample_values_2,jaccard_score,lazo_score,embedding_score,vote_count,confidence
pair_id,,,,,,,,,,,
0,i4gi-tjb9,borough,"Bronx, Brooklyn, Manhattan, Queens, Staten Island",kb2e-tjy3,borough,"Bronx, Brooklyn, Manhattan, Queens, Staten Island",1.0000,1.0000,1.0000,3,high
1,ci36-d7ea,survey_pp_se,"0.51, 0.64, 0.65, 0.66, 0.69",kkng-ugna,survey_pp_ct,"0.64, 0.66, 0.78, 0.8, 0.86",1.0000,0.7600,0.8543,3,high
2,phvi-damg,borough,"BRONX, BROOKLYN, MANHATTAN, QUEENS, STATEN ISLAND",rc2t-8fid,borough,"Bronx, Brooklyn, Manhattan, Queens, Staten Island",1.0000,1.0000,1.0000,3,high
3,922w-z7da,borough,"Bronx, Brooklyn, Manhattan, Queens, Staten Island",avhb-5jhc,borough,BROOKLYN,1.0000,1.0000,1.0000,3,high
4,5ery-qagt,city,"Bronx, Brooklyn, New York, Ridgewood, Whitestone",n3p6-zve2,city,"Bronx, Brooklyn, Cambria Heights, Elmhurst, Fa...",1.0000,0.6429,1.0000,3,high
5,3spy-rjpw,borough,"BRONX, BROOKLYN, MANHATTAN, QUEENS, STATEN IS",at6q-ktig,borough,Queens,1.0000,1.0000,1.0000,3,high
6,jdn9-td9w,borough,"BRONX, BROOKLYN, MANHATTAN, QUEENS",vd7c-qjsx,borough,"Bronx, Brooklyn",1.0000,0.8000,1.0000,3,high
7,tzwr-vksx,borough,"BRONX, BROOKLYN, MANHATTAN",ur7y-ziyb,borough,"Bronx, Brooklyn, Manhattan, No Verifiable Asso...",1.0000,1.0000,1.0000,3,high
8,6rrm-vxj9,borough,"Brooklyn, Manhattan, the Bronx",bkui-39n8,borough,"BRONX, BROOKLYN, MANHATTAN, QUEENS",1.0000,0.8000,1.0000,3,high


In [75]:
ground_truth['confidence'].value_counts()

confidence
medium      100
high         50
low          50
negative     50
Name: count, dtype: int64